In [57]:
import numpy as np
from pyscf import gto, scf, mp, cc

lno_num = 4
lno_list = [3e-4,1e-4,3e-5,1e-5]
lno_thresh = lno_list[lno_num-1]

####  test H2 monomers ####
a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 2 # spin per monomer
elmt = 'O'
unit = 'B'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=40000,
            )

mf = scf.UHF(mol).density_fit()
mf.max_cycle = 1
mf.kernel()

# stable = False
# while not stable:
#     print(f'mean-field stability test')
#     if not stable:
#         mo_i, _, stable,_ = mf.stability(return_status=True)
#         dm = mf.make_rdm1(mo_i,mf.mo_occ)
#         mf.kernel(dm0=dm)
#     elif stable:
#         print(f'UHF Energy: {mf.e_tot}, stability {stable}')
#         break

# mymp = mp.MP2(mf).set_frozen()
# mymp.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

print(f"HF   : {mf.e_tot}")
# print(f"MP2  : {mymp.e_tot}")
print(f"CCSD : {mycc.e_tot}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Thu Jun 25 12:54:00 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = B
[INPUT] Symbol           X                Y                Z      uni

In [58]:
print(f"HF per M   : {mf.e_tot/nc}")
print(f"CCSD per M : {mycc.e_tot/nc}")

HF per M   : -148.9587952587668
CCSD per M : -149.03984563307785


In [59]:
import jax
jax.config.update("jax_enable_x64", True)
import opt_einsum as oe

In [60]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-8)


Preparing AFQMC calculation
Calculating Cholesky integrals
Find Density Fit Teonsers in MF object
Integrals will be built by DF Tensors
Building JK matrix
Alpha Cholesky shape: (51, 8, 8) 
 Beta Cholesky shape: (51, 8, 8) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [7 5]
Number of basis functions:  8
Number of Cholesky vectors: 51


In [61]:
options = {
           'n_prop_steps': 50,
           'n_blocks': 10,
           'n_walkers': 300,
           'max_memory': 2000,
           'mix_precision': False,
           'n_batch': 1,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

In [62]:
import numpy as np
from jax import random
from jax import numpy as jnp

from afqmc import config, prep, sampling

from functools import partial

print = partial(print, flush=True)

# print("\nAFQMC Started")
# prep.print_start()
# config.setup_jax()

ham_data, ham, prop, trial, wave_data, sampler, options = (prep.init_afqmc(options=options))

wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)


Load system from Integral File
Maximum memory per walker:            6.67 MB
Maximum number of Cholesky per chunk: 3413
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         51
Number of padding Cholesky:           0

QMC System
Number of electrons: (7, 5)
Spin Multiplicity:   2
Number of orbitals:  8
Number of Chol:      51

QMC Parameters
n_prop_steps    -         50
n_blocks        -         10
n_walkers       -        300
max_memory      -       2000
mix_precision   -      False
n_batch         -          1
seed            -         17
walker_type     -        uhf
trial           -   upt2ccsd
dt              -      0.005
n_exp_terms     -          6
eql_time        -         20
max_error       -        0.0
nchol_chunk     -         51
free_projection -      False

Initalize QMC walkers by HF


In [ ]:
from afqmc import slater_tools
from jax import jit, jvp, lax
from jax import numpy as jnp

@partial(jit, static_argnums=0)
def get_fock(nocc, h1, chol):
    nocca, noccb = nocc
    h1a, h1b = h1
    chola, cholb = chol
    la = oe.contract('gii->g', chola[:,:nocca,:nocca], backend="jax")
    lb = oe.contract('gii->g', cholb[:,:noccb,:noccb], backend="jax")
    jeffa = oe.contract('gpq,g->pq', chola, la+lb, backend="jax")
    jeffb = oe.contract('gpq,g->pq', cholb, la+lb, backend="jax")
    keffa = oe.contract('gpj,gjq->pq', chola[:,:,:nocca], chola[:,:nocca,:], backend="jax")
    keffb = oe.contract('gpj,gjq->pq', cholb[:,:,:noccb], cholb[:,:noccb,:], backend="jax")
    focka = h1a + jeffa - keffa
    fockb = h1b + jeffb - keffb
    return (focka, fockb)

@jit
def u_energy_corr(bra, ket, e0, fock, chol):
    '''
    calculate the correlation energy of the Hamiltonian
    '''

    chola, cholb = chol
    if len(chola.shape) == 3:
        chola = chola.reshape(1,*chola.shape)
    if len(cholb.shape) == 3:
        cholb = cholb.reshape(1,*cholb.shape)

    norba, nocca = ket[0].shape 
    norbb, noccb = ket[1].shape
    rot_focka = fock[0][:nocca,nocca:]
    rot_fockb = fock[1][:noccb,noccb:]
    rot_chola = chola[:,:,:nocca,nocca:] # shape(nchunk,nchol_chunk,nocc,nvir)
    rot_cholb = cholb[:,:,:noccb,noccb:]

    gfa = (ket[0].dot(jnp.linalg.inv(ket[0][:nocca,:]))).T
    gfb = (ket[1].dot(jnp.linalg.inv(ket[1][:noccb,:]))).T
    gfa = gfa[:nocca, nocca:]
    gfb = gfb[:noccb, noccb:]
    e1a = oe.contract('ia,ia->', gfa, rot_focka, backend="jax")
    e1b = oe.contract('ia,ia->', gfb, rot_fockb, backend="jax")
    e1 = e1a + e1b

    # two body energy — chunked over Cholesky auxiliary index
    # nchol_chunk = self.nchol_chunk
    # nchol = rot_chola.shape[0]
    # nchunks = -(-nchol // nchol_chunk)
    # npad = nchunks * nchol_chunk - nchol
    # rot_chola = jnp.concatenate([rot_chola, jnp.zeros((npad, nocca, norba))], axis=0)
    # rot_cholb = jnp.concatenate([rot_cholb, jnp.zeros((npad, noccb, norbb))], axis=0)
    # rot_chola = rot_chola.reshape(nchunks, nchol_chunk, nocca, norba)
    # rot_cholb = rot_cholb.reshape(nchunks, nchol_chunk, noccb, norbb)

    def scan_chunk(carry, x):
        rot_chola_c, rot_cholb_c = x
        # explicit contraction within the chunk (g is chunk-local aux index)
        lga = oe.contract('gia,ja->gij', rot_chola_c, gfa, backend="jax")
        lgb = oe.contract('gia,ja->gij', rot_cholb_c, gfb, backend="jax")
        tr_lga = oe.contract('gii->g', lga, backend="jax")
        tr_lgb = oe.contract('gii->g', lgb, backend="jax")
        e2aa = oe.contract('g,g->', tr_lga, tr_lga, backend="jax") \
            - oe.contract('gij,gji->', lga, lga, backend="jax")
        e2ab = oe.contract('g,g->', tr_lga, tr_lgb, backend="jax")
        e2ba = oe.contract('g,g->', tr_lgb, tr_lga, backend="jax")
        e2bb = oe.contract('g,g->', tr_lgb, tr_lgb, backend="jax") \
            - oe.contract('gij,gji->', lgb, lgb, backend="jax")
        carry += 0.5 * (e2aa + e2ab + e2ba + e2bb)
        return carry, 0.0

    e2, _ = lax.scan(scan_chunk, 0.0, (rot_chola, rot_cholb))

    return e0 + e1 + e2, e1, e2

@jit
def u_energy_corr_with_f(bra, ket, e0, fock, chol):
    '''
    correlation energy E_corr = <bra|H-E0|ket>/<bra|ket> 
    '''
    chola, cholb = chol
    if len(chola.shape) == 3:
        chola = chola.reshape(1,*chola.shape)
    if len(cholb.shape) == 3:
        cholb = cholb.reshape(1,*cholb.shape)

    norba, nocca = ket[0].shape
    norbb, noccb = ket[1].shape
    rot_focka = fock[0][:nocca,nocca:]
    rot_fockb = fock[1][:noccb,noccb:]
    rot_chola = chola[:,:,:nocca,nocca:] # shape(nchol,nocc,nvir)
    rot_cholb = cholb[:,:,:noccb,noccb:]

    greena = (ket[0].dot(jnp.linalg.inv(ket[0][:nocca,:]))).T
    greenb = (ket[1].dot(jnp.linalg.inv(ket[1][:noccb,:]))).T
    greena = greena[:nocca, nocca:]
    greenb = greenb[:noccb, noccb:]

    e1a = oe.contract('ia,ia->', greena, rot_focka, backend="jax")
    e1b = oe.contract('ia,ia->', greenb, rot_fockb, backend="jax")
    e1 = e1a + e1b # should be zero for mean-field solution bra

    def scan_chol(carry, x):
        rot_chola_c, rot_cholb_c = x
        lga_c = oe.contract('gia,ja->gij', rot_chola_c, greena, backend="jax")
        lgb_c = oe.contract('gia,ja->gij', rot_cholb_c, greenb, backend="jax")
        tr_lga_c = oe.contract('gii->g',lga_c, backend="jax")
        tr_lgb_c = oe.contract('gii->g',lgb_c, backend="jax")
        tr_lg_c = tr_lga_c + tr_lgb_c
        e_col_c = jnp.sum(tr_lg_c**2) / 2
        e_exc_c = (oe.contract('gij,gji->',lga_c,lga_c, backend="jax")
                    + oe.contract('gij,gji->',lgb_c,lgb_c, backend="jax")) / 2
        ecorr_c = e_col_c - e_exc_c
        carry += ecorr_c
        return carry, 0.0
    
    e2, _ = lax.scan(scan_chol, 0.0, (rot_chola, rot_cholb))

    return e0 + e1 + e2

In [80]:
h1 = ham_data["h1"]
norba = h1[0].shape[0]
norbb = h1[1].shape[1]
(chola, cholb) = ham_data["chol"]
chola = chola.reshape(-1,norba,norba)
cholb = cholb.reshape(-1,norbb,norbb)
v0_a = 0.5 * oe.contract("gik,gjk->ij", chola, chola, backend='jax')
v0_b = 0.5 * oe.contract("gik,gjk->ij", cholb, cholb, backend='jax')
h1mod_a = np.array(h1[0]) - v0_a
h1mod_b = np.array(h1[1]) - v0_b
init_walker = (prop_data["walkers"][0][0], prop_data["walkers"][1][0])

In [77]:
nocc = (init_walker[0].shape[1], init_walker[1].shape[1])
fock = get_fock(nocc, h1, (chola, cholb))

In [82]:
bra = wave_data["mo_coeff"]
ket = init_walker
chola = chola.reshape(1,*chola.shape)
cholb = cholb.reshape(1,*cholb.shape)
chol = (chola, cholb)
e01 = slater_tools.u_energy(bra, ket, h0, h1, chol)
e02, e1, e2 = u_energy_corr2(bra, ket, mf.e_tot, fock, chol)
print(e01)
print(e02, e1, e2)
print(mf.e_tot)

(-148.95879525828178+0j)
(-148.9587952587668+0j) 0j 0j
-148.9587952587668


In [87]:
ket_up = jnp.array(np.random.rand(*init_walker[0].shape)) + 1j*jnp.array(np.random.rand(*init_walker[0].shape))
ket_dn = jnp.array(np.random.rand(*init_walker[1].shape)) + 1j*jnp.array(np.random.rand(*init_walker[1].shape))
ket = (ket_up, ket_dn)
e01 = slater_tools.u_energy(bra, ket, h0, h1, chol)
e02, e12, e22 = u_energy_corr2(bra, ket, mf.e_tot, fock, chol)
e03, e13, e23 = u_energy_corr(bra, ket, mf.e_tot, fock, chol)
chola, cholb = chol
if len(chola.shape) == 4:
    nc, nchol_c, _, _ = chola.shape
    chola = chola.reshape(nc*nchol_c,*chola.shape[-2:])
if len(cholb.shape) == 4:
    nc, nchol_c, _, _ = cholb.shape
    cholb = cholb.reshape(nc*nchol_c,*cholb.shape[-2:])
e00 = slater_tools.u_energy_ad(bra, ket, h0, (h1mod_a, h1mod_b), (chola,cholb))
print(e00)
print(e01)
print(e02, e12, e22)
print(e03, e13, e23)

(-148.897976505747+0.14120399935813066j)
(-148.89797650574698+0.1412039993581447j)
(-148.897976506232+0.14120399935814043j) (-0.04955140714031577-0.011088085762521419j) (0.11037015967512075+0.15229208512066184j)
(-148.897976506232+0.14120399935814026j) (-0.04955140714031577-0.011088085762521419j) (0.11037015967512075+0.15229208512066167j)
